# LFM3-VL-2B — image and short video on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/lfm3-colab-notebooks-103d/colab/LFM3-VL-2B-vision.ipynb)

[`LiquidAI/LFM3-VL-2B`](https://huggingface.co/LiquidAI/LFM3-VL-2B) is Liquid AI's **vision-language** LFM3 (~1.92B, `lfm3_vl`). It takes text, images, and **video** (sampled at **2 fps**) and answers in text. This is the first LFM vision model with video input. It is **not** the audio, omni, or d1 decision checkpoint.

The card's Transformers quick start is `transformers>=5.21.0` plus `trust_remote_code`. Native LFM3-VL support is still landing in [`huggingface/transformers#50036`](https://github.com/huggingface/transformers/pull/50036).

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-VL-2B/blob/main/LICENSE) (`license: other`, `lfm1.0`).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. Load **float16** + **SDPA**. Weights are about 3.8 GB. Image questions should sit around 5–8 GB. Video is capped at **8 frames / 2 fps** (the processor default is up to 768 frames — that will OOM a T4). `av` writes and decodes the clip if `torchcodec` is missing.

**Links:** [model card](https://huggingface.co/LiquidAI/LFM3-VL-2B) · [blog](https://www.liquid.ai/blog/lfm3) · [announcement](https://x.com/liquidai/status/2115128329003303107) · [collection](https://huggingface.co/collections/LiquidAI/lfm3) · [GGUF](https://huggingface.co/LiquidAI/LFM3-VL-2B-GGUF)


In [ ]:
%pip install -q -U "transformers>=5.21.0" accelerate
%pip install -q pillow torchvision av

# Card pin: transformers>=5.21.0, torch, torchvision, pillow, trust_remote_code.
# Do not reinstall torch. av writes the short T4 video and is the decode fallback
# when torchcodec is not available on Colab.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 1. Check the T4 and pick a dtype


In [ ]:
import os
import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.21.0"), (
    f"transformers {transformers.__version__} is loaded; this notebook needs 5.21.0+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# The cards default to bfloat16 on GPU. A T4 has no native BF16, so use float16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.float16
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")
if major >= 8:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


## 2. Load LFM3-VL-2B

`AutoModelForImageTextToText` + `AutoProcessor`, as on the [card](https://huggingface.co/LiquidAI/LFM3-VL-2B). Sampling: `temperature=0.1`, `top_k=50`, `repetition_penalty=1.1`. The card's `max_new_tokens=16384` is for long reasoning; T4 demos use **256**.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "LiquidAI/LFM3-VL-2B"

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_vlm(attn):
    return AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
        device_map="auto",
        trust_remote_code=True,
    ).eval()


ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_vlm(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load LFM3-VL-2B") from last_error

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print("attention:", ATTN, "| dtype:", getattr(model, "dtype", DTYPE))
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


def answer(messages, max_new_tokens=256, **template_kwargs):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        **template_kwargs,
    ).to(model.device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=model.dtype)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.1,
            top_k=50,
            repetition_penalty=1.1,
            max_new_tokens=max_new_tokens,
        )
    text = processor.batch_decode(
        generated[:, inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    )[0]
    print(
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )
    return text


## 3. Demo 1 — image question

Public COCO val image (two cats on a sofa), also used on the d1 cards. The VL card's own example uses `https://placecats.com/300/200`.


In [ ]:
from IPython.display import display
from transformers.image_utils import load_image

IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image(IMAGE_URL)
display(image.resize((320, 240)))

print(answer([
    {
        "role": "user",
        "content": [
            {"type": "image", "url": IMAGE_URL},
            {"type": "text", "text": "Describe this image in two sentences. How many cats are there?"},
        ],
    }
]))


## 4. Demo 2 — short video (2 fps, 8 frames)

The processor samples video at **2 fps** (`min_frames=4`, card default `max_frames=768`). A T4 cannot hold hundreds of native-resolution frames, so this cell writes an 8-frame, 320×320 clip of a moving square and also passes `max_frames=8`. Raise the cap on a larger GPU.


In [ ]:
from pathlib import Path

import av
import numpy as np
from IPython.display import Video, display
from PIL import Image

VIDEO_PATH = Path("lfm3_vl_moving_square.mp4")
N_FRAMES = 8
FPS = 2
SIZE = 320

container = av.open(str(VIDEO_PATH), mode="w")
stream = container.add_stream("mpeg4", rate=FPS)
stream.width = SIZE
stream.height = SIZE
stream.pix_fmt = "yuv420p"
preview = []
for i in range(N_FRAMES):
    frame = np.full((SIZE, SIZE, 3), (24, 48, 80), dtype=np.uint8)
    x = int((i / max(N_FRAMES - 1, 1)) * (SIZE - 64))
    frame[128:192, x : x + 64] = (240, 200, 40)
    preview.append(Image.fromarray(frame).resize((160, 160)))
    vf = av.VideoFrame.from_ndarray(frame, format="rgb24")
    for packet in stream.encode(vf):
        container.mux(packet)
for packet in stream.encode():
    container.mux(packet)
container.close()

print(f"wrote {VIDEO_PATH} ({N_FRAMES} frames @ {FPS} fps)")
display(*preview[:4])
display(Video(str(VIDEO_PATH), embed=True, width=320))

print(answer(
    [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": str(VIDEO_PATH)},
                {"type": "text", "text": "What moves in this clip, and in which direction?"},
            ],
        }
    ],
    videos_kwargs={"max_frames": N_FRAMES, "fps": float(FPS), "min_frames": 4},
))


## Optional GGUF

[`LiquidAI/LFM3-VL-2B-GGUF`](https://huggingface.co/LiquidAI/LFM3-VL-2B-GGUF) ships `Q4_K_M` (~1.1 GB) plus an mmproj. The card says llama.cpp LFM3-VL support is still in review ([ggml-org/llama.cpp#31252](https://github.com/ggml-org/llama.cpp/pull/31252)) **without video**. Transformers is the primary path in this notebook.


## Caveats

- **Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-VL-2B/blob/main/LICENSE) (`license: other`).
- **T4 video cap:** 8 frames at 2 fps. The processor will happily sample hundreds of frames; that OOMs 15 GB.
- **Install:** `transformers>=5.21.0` + `trust_remote_code`. Native LFM3-VL is still [PR #50036](https://github.com/huggingface/transformers/pull/50036).
- **HF token:** public weights. Cancel Colab `HF_TOKEN` prompts.
- **Not executed here:** no GPU in the agent VM. Full T4 `generate()` was not run.
- **Do not merge from this agent.**
